# Vision Transformer (ViT) per la Ricostruzione di Immagini

In questo notebook implementiamo un **Vision Transformer (ViT)** per la ricostruzione di immagini degradate.
Il ViT è un'architettura che applica i meccanismi di *self-attention* dei Transformer, originariamente pensati per il Natural Language Processing, direttamente alle immagini.

L'idea chiave è suddividere l'immagine in **patch** (porzioni rettangolari), proiettarle in uno spazio di embedding e poi processarle con blocchi Transformer.

---

## 1 · Patch Embedding (`PatchEmbed`)

Il primo passaggio fondamentale del ViT è il **Patch Embedding**: l'immagine viene divisa in patch non sovrapposte di dimensione $P \times P$ e ogni patch viene proiettata linearmente in un vettore di dimensione `embed_dim`.

In pratica, questa operazione è realizzata con una **convoluzione 2D** con `kernel_size = patch_size` e `stride = patch_size`: ogni filtro "guarda" esattamente una patch e produce un singolo valore per canale di uscita.

**Input:** tensore `(B, C, H, W)` — batch di immagini.  
**Output:** tensore `(B, N, d)` dove $N = (H/P) \times (W/P)$ è il numero di patch e $d$ è la dimensione dell'embedding, più la tupla delle dimensioni della griglia `(H/P, W/P)` (utile per ricostruire la struttura spaziale nel decoder).


In [ ]:
import torch
from torch import nn

class PatchEmbed(nn.Module):
    def __init__(self, in_ch=1, embed_dim=128, patch_size=8):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(
            in_channels=in_ch,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        z = self.proj(x)  # (B, d, H/P, W/P)
        B, d, H_p, W_p = z.shape
        z = z.flatten(2).transpose(1, 2)  # (B, N, d)
        return z, (H_p, W_p)

## 2 · Blocco Encoder del Transformer (`TransformerEncoderBlock`)

Questo è il componente centrale del ViT. Ogni blocco segue lo schema classico del **Transformer Encoder** (Vaswani et al., 2017) con normalizzazione *pre-norm*:

$$z' = z + \text{MultiHeadAttention}(\text{LayerNorm}(z))$$
$$z_{out} = z' + \text{MLP}(\text{LayerNorm}(z'))$$

I sotto-moduli principali sono:

| Componente | Descrizione |
|---|---|
| **`LayerNorm`** | Normalizzazione per canale, stabilizza il training |
| **`MultiheadAttention`** | Meccanismo di *self-attention* multi-testa: ogni token (patch) può "guardare" tutte le altre patch per catturare dipendenze globali |
| **MLP (Feed-Forward Network)** | Due layer lineari con attivazione **GELU** in mezzo, espande e poi comprime la rappresentazione |

Le **connessioni residue** (`z + ...`) sono fondamentali per permettere il flusso del gradiente e facilitare l'addestramento di reti profonde.

> **Nota:** a differenza delle CNN, la self-attention ha un *receptive field globale* fin dal primo layer: ogni patch può interagire con tutte le altre, indipendentemente dalla distanza spaziale.


In [ ]:

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim),
        )

    def forward(self, z):
        h = self.norm1(z)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        z = z + attn_out
        z = z + self.mlp(self.norm2(z))
        return z

## 3 · Modello completo: `ViTReconstructor`

Questa classe assembla tutti i componenti in un modello end-to-end per la **ricostruzione di immagini**:

```
Immagine degradata (B,1,H,W)
       │
       ▼
  ┌─────────────┐
  │ PatchEmbed   │  → (B, N, d)   suddivisione in patch + proiezione lineare
  └─────────────┘
       │
       ▼  + Positional Embedding (learnable)
  ┌─────────────┐
  │ Encoder      │  × depth blocchi Transformer
  │ Transformer  │  → (B, N, d)   self-attention globale
  └─────────────┘
       │
       ▼  reshape → (B, d, H/P, W/P)
  ┌─────────────┐
  │ Decoder CNN  │  ConvTranspose2d + Conv2d → (B, 1, H, W)
  └─────────────┘
       │
       ▼  + residual (input originale)
    Output (B,1,H,W)
```

### Dettagli delle componenti:

- **Positional Embedding** (`pos_embed`): è un parametro *learnable* di shape `(1, N, d)`. Viene sommato agli embedding delle patch per iniettare informazione sulla **posizione spaziale**, che altrimenti andrebbe persa nella tokenizzazione.

- **Encoder**: una sequenza di `depth` blocchi `TransformerEncoderBlock`, che raffinano progressivamente la rappresentazione tramite self-attention.

- **Decoder**: un blocco puramente convoluzionale che riporta la feature map dalla risoluzione ridotta `(H/P, W/P)` alla risoluzione originale `(H, W)` tramite una `ConvTranspose2d` (deconvoluzione) seguita da layer `Conv2d` con attivazione `ReLU`.

- **Connessione residua globale**: l'output finale è `input + decoder(encoder(input))`. Questo significa che la rete impara solo il **residuo** (la differenza tra immagine pulita e degradata), semplificando enormemente il task di apprendimento.


In [ ]:
class ViTReconstructor(nn.Module):
    def __init__(
        self,
        img_size=256,
        patch_size=8,
        in_ch=1,
        out_ch=1,
        embed_dim=128,
        depth=4,
        num_heads=4,
        mlp_dim=256,
    ):
        super().__init__()

        assert img_size % patch_size == 0, 'img_size must be divisible by patch_size.'
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2

        self.patch_embed = PatchEmbed(in_ch=in_ch, embed_dim=embed_dim, patch_size=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))

        self.encoder = nn.Sequential(*[
            TransformerEncoderBlock(embed_dim=embed_dim, num_heads=num_heads, mlp_dim=mlp_dim)
            for _ in range(depth)
        ])

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels=embed_dim,
                out_channels=embed_dim // 2,
                kernel_size=patch_size,
                stride=patch_size,
            ),
            nn.ReLU(),
            nn.Conv2d(embed_dim // 2, embed_dim // 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(embed_dim // 2, embed_dim // 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(embed_dim // 4, out_ch, kernel_size=1),
        )

    def forward(self, x):
        residual = x
        z, (H_p, W_p) = self.patch_embed(x)
        z = z + self.pos_embed[:, :z.shape[1], :]
        z = self.encoder(z)

        feat = z.transpose(1, 2).reshape(x.shape[0], -1, H_p, W_p)
        out = self.decoder(feat)
        return residual + out


## 4 · Test di funzionamento del modello

Verifichiamo che il modello sia costruito correttamente creando un'istanza di `ViTReconstructor` e passando un batch di input casuale.

Ci aspettiamo che l'output abbia la **stessa forma dell'input** `(B, 1, 256, 256)`, dato che il modello è progettato per la ricostruzione (l'immagine di uscita deve avere le stesse dimensioni di quella in ingresso).


In [ ]:
vit_model = ViTReconstructor(img_size=256, patch_size=8, in_ch=1, out_ch=1, embed_dim=128, depth=4, num_heads=4, mlp_dim=256)
x = torch.rand(2, 1, 256, 256)
y = vit_model(x)

print('Input shape:', tuple(x.shape))
print('Output shape:', tuple(y.shape))

## 5 · Setup dell'ambiente: librerie, dataset e operatore di degradazione

In questa cella prepariamo tutto il necessario per l'addestramento:

1. **Importazione delle librerie**: `matplotlib` per la visualizzazione, `torch` e moduli correlati per il deep learning, `tqdm` per le barre di progresso, `PIL` per la lettura delle immagini.

2. **Localizzazione delle risorse**: il codice cerca automaticamente nella directory corrente (e nelle sue parent) le cartelle `Mayo/` (dataset) e `weights/` (pesi salvati), e il pacchetto locale `IPPy` che contiene gli operatori di imaging.

3. **Selezione del device**: viene scelto automaticamente il dispositivo di calcolo migliore disponibile (`cuda` > `mps` > `cpu`).

4. **Funzione di rumore gaussiano**: `gaussian_noise(y, noise_level)` genera rumore additivo gaussiano con norma relativa controllata dal parametro `noise_level`.

5. **`MayoDataset`**: un dataset PyTorch custom che carica le immagini CT dal dataset Mayo, le converte in scala di grigi e le ridimensiona a `256×256`.

6. **Operatore di degradazione `K`**: un operatore di **blurring** (sfocatura) di tipo *motion blur* con kernel `9×9` e angolo di 20°, che simula la degradazione dell'immagine.

> **Output atteso:** nessun output diretto; la cella prepara `train_loader`, `test_loader`, il `device` e l'operatore `K` per le celle successive.


In [ ]:
import glob
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

here = Path.cwd().resolve()
for base in (here, *here.parents):
    if (base / 'weights').exists() and (base / 'Mayo').exists():
        book_root = base
        break
else:
    raise FileNotFoundError('Could not locate the course root containing Mayo and weights.')

for base in (here, *here.parents):
    if (base / 'IPPy').exists():
        ippy_root = base / 'IPPy'
        break
else:
    raise FileNotFoundError('Could not locate the local IPPy package.')

operators_spec = importlib.util.spec_from_file_location('course_ippy_operators', ippy_root / 'operators.py')
operators = importlib.util.module_from_spec(operators_spec)
operators_spec.loader.exec_module(operators)

weights_dir = book_root / 'weights'
weights_dir.mkdir(exist_ok=True)

def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    try:
        if torch.backends.mps.is_available():
            return 'mps'
    except AttributeError:
        pass
    return 'cpu'

def gaussian_noise(y, noise_level):
    e = torch.randn_like(y, device=y.device)
    return e / torch.norm(e) * torch.norm(y) * noise_level

class MayoDataset(Dataset):
    def __init__(self, data_path, data_shape):
        super().__init__()
        self.data_path = data_path
        self.data_shape = data_shape
        self.fname_list = glob.glob(f'{data_path}/*/*.png')

    def __len__(self):
        return len(self.fname_list)

    def __getitem__(self, idx):
        img_path = self.fname_list[idx]
        x = Image.open(img_path).convert('L')
        x = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize(self.data_shape),
        ])(x)
        return x

device = get_device()
train_dataset = MayoDataset(data_path=str(book_root / 'Mayo' / 'train'), data_shape=256)
test_dataset = MayoDataset(data_path=str(book_root / 'Mayo' / 'test'), data_shape=256)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

K = operators.Blurring(
    img_shape=(256, 256),
    kernel_type='motion',
    kernel_size=9,
    motion_angle=20,
)

## 6 · Inizializzazione del modello, ottimizzatore e funzione di loss

Creiamo un'istanza del `ViTReconstructor` con i seguenti iperparametri:

| Parametro | Valore | Significato |
|---|---|---|
| `img_size` | 256 | Dimensione dell'immagine in pixel |
| `patch_size` | 8 | Lato delle patch → $N = (256/8)^2 = 1024$ patch |
| `in_ch` / `out_ch` | 1 | Immagini in scala di grigi (1 canale) |
| `embed_dim` | 128 | Dimensione dello spazio di embedding |
| `depth` | 4 | Numero di blocchi Transformer nell'encoder |
| `num_heads` | 4 | Numero di teste nell'attenzione multi-testa |
| `mlp_dim` | 256 | Dimensione della hidden layer nel feed-forward |

- **Ottimizzatore**: Adam con learning rate $10^{-3}$.
- **Loss**: Mean Squared Error (MSE), che misura la distanza pixel-per-pixel tra immagine ricostruita e ground truth.

> Il seed `torch.manual_seed(0)` garantisce la **riproducibilità** dell'inizializzazione dei pesi.


In [ ]:
torch.manual_seed(0)
vit_model = ViTReconstructor(
    img_size=256,
    patch_size=8,
    in_ch=1,
    out_ch=1,
    embed_dim=128,
    depth=4,
    num_heads=4,
    mlp_dim=256,
).to(device)
optimizer = torch.optim.Adam(vit_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

## 7 · Ciclo di addestramento (Training Loop)

Il training segue il paradigma supervisionato classico per problemi di ricostruzione:

Per ogni batch di immagini `x` dal dataset:
1. Si applica l'operatore di degradazione `K` per ottenere l'immagine sfocata `y = K(x)`.
2. Si aggiunge **rumore gaussiano** con `noise_level = 0.01` a `y`.
3. Il modello ViT produce la ricostruzione: `prediction = ViT(y)`.
4. Si calcola la **loss MSE** tra `prediction` e l'immagine originale `x`.
5. Si esegue la backpropagation e l'aggiornamento dei pesi.

Sono previste **10 epoche** di addestramento. Al termine, i pesi vengono salvati in `weights/ViT.pth`.

Le barre di progresso `tqdm` mostrano:
- La **batch loss** corrente.
- La **loss media** sull'epoca.

> **Output atteso:** barre di progresso per ogni epoca e un messaggio di conferma del salvataggio dei pesi.


In [ ]:
num_epochs = 10
noise_level = 0.01
history = []
weights_path = weights_dir / 'ViT.pth'

epoch_progress = tqdm(range(num_epochs), desc='Training Progress', unit='epoch')
for epoch in epoch_progress:
    vit_model.train()
    epoch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}', leave=True)

    for step, x_batch in enumerate(progress_bar, start=1):
        x_batch = x_batch.to(device)

        with torch.no_grad():
            y_batch = K(x_batch)
            y_batch = y_batch + gaussian_noise(y_batch, noise_level=noise_level)

        prediction = vit_model(y_batch)
        loss = loss_fn(prediction, x_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix(batch_loss=f'{loss.item():.4f}', avg_loss=f'{epoch_loss / step:.4f}')

    avg_loss = epoch_loss / len(train_loader)
    history.append(avg_loss)
    epoch_progress.set_postfix(last_loss=f'{avg_loss:.6f}')

torch.save(vit_model.state_dict(), weights_path)
print(f'Saved ViT weights to: {weights_path}')

## 8 · Visualizzazione dei risultati

In questa ultima cella carichiamo i pesi addestrati e **visualizziamo** i risultati della ricostruzione su un esempio dal test set. Vengono mostrate tre immagini affiancate:

1. **Immagine originale** (`x`): il ground truth, l'immagine CT pulita.
2. **Immagine degradata** (`y`): il risultato dell'applicazione dell'operatore di blurring + rumore.
3. **Ricostruzione ViT**: l'output del modello, che idealmente dovrebbe essere vicino all'immagine originale.

Questo confronto visivo permette di valutare qualitativamente la capacità del ViT di rimuovere la degradazione e ricostruire i dettagli dell'immagine.

> **Output atteso:** un'immagine con tre sottofigure affiancate (originale, degradata, ricostruita).


In [ ]:
reloaded_vit = ViTReconstructor(
    img_size=256,
    patch_size=8,
    in_ch=1,
    out_ch=1,
    embed_dim=128,
    depth=4,
    num_heads=4,
    mlp_dim=256,
)
reloaded_vit.load_state_dict(torch.load(weights_path, map_location='cpu', weights_only=True))
reloaded_vit = reloaded_vit.to(device)
reloaded_vit.eval()

with torch.no_grad():
    x_test = next(iter(test_loader))[0:1].to(device)
    y_test = K(x_test)
    y_test = y_test + gaussian_noise(y_test, noise_level=noise_level)
    x_pred = reloaded_vit(y_test)

plt.figure(figsize=(15, 4))
plt.subplot(1, 4, 1)
plt.imshow(x_test.cpu().squeeze(), cmap='gray')
plt.title('Ground truth')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(y_test.cpu().squeeze(), cmap='gray')
plt.title('Measurement')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(x_pred.cpu().squeeze(), cmap='gray')
plt.title('Reloaded ViT')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.plot(history)
plt.title('Training loss')
plt.xlabel('Epoch')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()